# TruthLens Audio -- v6 notebook

v5 (75% held-out accuracy) was strong on deepfake detection (87% recall) but weak on
genuine-voice recognition for a brand-new speaker (40% recall). Diagnosed cause: each
realistic-voice-changing script folder has 1 `original` file but 3 `synthetic_*` files, so
even with the same oversample factor applied to both, genuine-labeled realistic rows ended
up with 1/3 the absolute weight of deepfake-labeled ones in training -- an accident of the
folder structure, not a deliberate choice. Confirmed by testing v5 on its own training
speakers' genuine files: 15/15 correct, high confidence -- so it's not a fundamental
genuine-vs-ai_generated confusion, just an underweighted signal for a new speaker.

v6 fixes this: same combine + gentle fine-tune-from-v2 recipe as v5, but genuine-labeled
realistic rows get a higher oversample factor (60x vs deepfake's 20x) so both land at
roughly the same absolute row count.

Run cells top to bottom. Needs the realistic eval zip re-uploaded again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

PROJECT_DIR = '/content/drive/MyDrive/TruthLens_Audio'
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

LABELS = ['genuine', 'ai_generated', 'deepfake']
LABEL_TO_IDX = {l: i for i, l in enumerate(LABELS)}

MANIFEST_DIR = f'{PROJECT_DIR}/manifests_v2'   # v2's original manifest, needed as df_v2_base
export_dir = f'{PROJECT_DIR}/export_v2'
export_dir_v3 = f'{PROJECT_DIR}/export_v3'

RUN_NAME_V4 = 'v4'
CKPT_DIR_V4 = f'{PROJECT_DIR}/checkpoints_{RUN_NAME_V4}'

RUN_NAME_V5 = 'v5'
CKPT_DIR_V5 = f'{PROJECT_DIR}/checkpoints_{RUN_NAME_V5}'

RUN_NAME_V6 = 'v6'
CKPT_DIR_V6 = f'{PROJECT_DIR}/checkpoints_{RUN_NAME_V6}'
LOG_DIR_V6 = f'{PROJECT_DIR}/logs_{RUN_NAME_V6}'
MANIFEST_DIR_V6 = f'{PROJECT_DIR}/manifests_{RUN_NAME_V6}'
for d in [CKPT_DIR_V6, LOG_DIR_V6, MANIFEST_DIR_V6]:
    os.makedirs(d, exist_ok=True)

print('Setup complete.')


In [ ]:
!pip install -q transformers accelerate soundfile librosa scikit-learn kaggle pandas tqdm av


In [ ]:
import os, shutil
from pathlib import Path

# Safety net only -- restores below should already provide everything.
kaggle_cred_drive = Path(f'{PROJECT_DIR}/kaggle.json')

if not kaggle_cred_drive.exists():
    print('No cached Kaggle credential found in Drive -- please upload one now.')
    print('(Kaggle -> Settings -> API -> Create New Token)')
    from google.colab import files
    uploaded = files.upload()
    assert uploaded, 'Expected a file to be uploaded'
    uploaded_name = next(iter(uploaded))
    shutil.copy(uploaded_name, kaggle_cred_drive)
    print('Cached to Drive for next time.')

raw = kaggle_cred_drive.read_text().strip()

if raw.startswith('KGAT_'):
    os.environ['KAGGLE_API_TOKEN'] = raw
    legacy_path = Path(os.path.expanduser('~/.kaggle/kaggle.json'))
    if legacy_path.exists():
        legacy_path.unlink()
    print('Detected new-style KGAT_ token -> set KAGGLE_API_TOKEN env var.')
else:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy(kaggle_cred_drive, os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    print('Detected classic kaggle.json -> copied to ~/.kaggle/kaggle.json.')

print('Kaggle credentials ready.')


## Restore raw audio data from Drive backups

In [ ]:
import subprocess

backup_path = f'{PROJECT_DIR}/data_backup.tar.gz'
assert os.path.exists(backup_path), f'No backup found at {backup_path} -- nothing to restore.'

subprocess.run(['cp', backup_path, '/content/data_backup.tar.gz'], check=True)
subprocess.run(['tar', '-xzf', '/content/data_backup.tar.gz', '-C', '/content'], check=True)
print('Restored v2 raw audio data (ASVspoof + FakeAVCeleb + WaveFake) from Drive backup.')


In [ ]:
import subprocess, os

def kaggle_download(dataset_ref, dest_subdir):
    dest = f'{DATA_DIR}/{dest_subdir}'
    if os.path.exists(dest) and os.listdir(dest):
        print(f'[skip] {dest_subdir} already present at {dest}')
        return dest
    os.makedirs(dest, exist_ok=True)
    print(f'Downloading {dataset_ref} -> {dest} ...')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', dataset_ref, '-p', dest, '--unzip'], check=True)
    return dest

V4_DATA_BACKUP = f'{PROJECT_DIR}/v4_data_backup.tar.gz'
restored_from_backup = False
if os.path.exists(V4_DATA_BACKUP):
    print('Found existing v4 data backup on Drive -- restoring instead of re-downloading.')
    subprocess.run(['cp', V4_DATA_BACKUP, '/content/v4_data_backup.tar.gz'], check=True)
    subprocess.run(['tar', '-xzf', '/content/v4_data_backup.tar.gz', '-C', '/content'], check=True)
    restored_from_backup = True
    print('Restored.')

    # The backup was archived relative to DATA_DIR but extracts relative to /content -- one
    # directory level off. Symlink so DATA_DIR/<name> resolves to the real extracted files,
    # BEFORE kaggle_download's existence check runs, so it doesn't wrongly re-download.
    os.makedirs(DATA_DIR, exist_ok=True)
    for name in ['adarshsingh_adf', 'deepvoice', 'deepvoice_chunks']:
        real = f'/content/{name}'
        link = f'{DATA_DIR}/{name}'
        if os.path.exists(real) and not os.path.exists(link):
            os.symlink(real, link)
            print(f'Linked {link} -> {real}')

adf_dir = kaggle_download('adarshsingh0903/audio-deepfake-detection-dataset', 'adarshsingh_adf')
deepvoice_dir = kaggle_download('birdy654/deep-voice-deepfake-voice-recognition', 'deepvoice')
print('Done.')


## Helpers, model class, and the already-trained v2 model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import Wav2Vec2Model
import soundfile as sf
import librosa
import numpy as np
import pandas as pd
import glob
from pathlib import Path

EVAL_LABELS = ['genuine', 'ai_generated', 'deepfake']
TARGET_SR = 16000
WINDOW_SAMPLES = TARGET_SR * 4

def load_mono_16k(path):
    # soundfile can't open some containers (notably .m4a) -- fall back to librosa
    # (ffmpeg-backed) rather than silently failing.
    try:
        audio, sr = sf.read(path, always_2d=False)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != TARGET_SR:
            audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=TARGET_SR)
        return audio.astype(np.float32)
    except Exception:
        audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
        return audio.astype(np.float32)

def rms_normalize(audio, target_dbfs=-20.0):
    rms = np.sqrt(np.mean(audio ** 2)) + 1e-9
    target_rms = 10 ** (target_dbfs / 20)
    return audio * (target_rms / rms)

def fixed_window(audio, length, train=False):
    if len(audio) >= length:
        if train:
            start = np.random.randint(0, len(audio) - length + 1)
        else:
            start = (len(audio) - length) // 2
        return audio[start:start + length]
    pad = length - len(audio)
    left = pad // 2
    right = pad - left
    return np.pad(audio, (left, right), mode='reflect')

class AudioDataset(Dataset):
    def __init__(self, manifest_df, split, train):
        self.rows = manifest_df[manifest_df['split'] == split].reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        try:
            audio = load_mono_16k(row['path'])
            audio = rms_normalize(audio)
            audio = fixed_window(audio, WINDOW_SAMPLES, self.train)
        except Exception:
            audio = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
        return torch.from_numpy(audio), LABEL_TO_IDX[row['label']]

class AudioClassifier(nn.Module):
    def __init__(self, num_classes=3, freeze_feature_extractor=True):
        super().__init__()
        self.backbone = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base')
        if freeze_feature_extractor:
            self.backbone.feature_extractor._freeze_parameters()
        hidden = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, num_classes))

    def forward(self, waveform):
        out = self.backbone(waveform).last_hidden_state
        return self.head(out.mean(dim=1))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: no GPU detected. Runtime -> Change runtime type -> GPU, then re-run from here.')

eval_model = AudioClassifier(num_classes=3).to(device)
eval_model.load_state_dict(torch.load(f'{export_dir}/wav2vec2_deepfake_classifier.pt', map_location=device))
eval_model.eval()
print('Loaded v2 model on', device)


## Load v3, v4, and v5 from their already-trained checkpoints

In [ ]:
ft_model = AudioClassifier(num_classes=len(LABELS)).to(device)
ft_model.load_state_dict(torch.load(f'{export_dir_v3}/wav2vec2_deepfake_classifier.pt', map_location=device))
ft_model.eval()
print('Loaded v3 model from', export_dir_v3)

v4_model = AudioClassifier(num_classes=len(LABELS)).to(device)
v4_model.load_state_dict(torch.load(f'{CKPT_DIR_V4}/best.pt', map_location=device))
v4_model.eval()
print('Loaded v4 best checkpoint.')

v5_model = AudioClassifier(num_classes=len(LABELS)).to(device)
v5_state = torch.load(f'{CKPT_DIR_V5}/latest.pt', map_location=device)
v5_model.load_state_dict(v5_state['model'])   # latest.pt wraps the weights in a dict; v5 never
                                                # beat its own pre-training baseline, so it has
                                                # no best.pt -- latest.pt is its only checkpoint.
v5_model.eval()
print('Loaded v5 latest checkpoint.')


In [ ]:
def infer_with(m, path):
    audio = load_mono_16k(path)
    audio = rms_normalize(audio)
    audio = fixed_window(audio, WINDOW_SAMPLES)
    waveform = torch.from_numpy(audio).unsqueeze(0).to(device)
    m.eval()
    with torch.no_grad():
        probs = F.softmax(m(waveform), dim=-1).cpu().numpy()[0]
    return probs


## Rebuild adarshsingh_adf and DEEP-VOICE manifests (raw data already restored above)

In [ ]:
import glob
from pathlib import Path

top_level_dirs = sorted({Path(p).relative_to(adf_dir).parts[0]
                          for p in glob.glob(f'{adf_dir}/**/*.wav', recursive=True)})
print('Top-level folders found:', top_level_dirs)
for d in top_level_dirs:
    n = len(glob.glob(f'{adf_dir}/{d}/*.wav'))
    print(f'  {d}: {n} files')

adf_rows = []
for d in top_level_dirs:
    label = 'genuine' if 'real' in d.lower() else 'ai_generated'
    files = glob.glob(f'{adf_dir}/{d}/*.wav')
    for f in files:
        adf_rows.append({
            'path': f, 'label': label, 'source': f'adarshsingh_adf_{d}',
            'speaker_id': Path(f).stem, 'split_hint': None,
        })

adf_df_check = pd.DataFrame(adf_rows)
print(f'Dataset 1 rows: {len(adf_rows)}')
print(adf_df_check.groupby(['label', 'source']).size())
assert (adf_df_check['label'] == 'genuine').sum() > 0, (
    'No folder matched as genuine -- check the folder names printed above.'
)


In [ ]:
import soundfile as sf
from tqdm import tqdm

DEEPVOICE_CHUNK_DIR = f'{DATA_DIR}/deepvoice_chunks'
os.makedirs(DEEPVOICE_CHUNK_DIR, exist_ok=True)
CHUNK_SAMPLES = TARGET_SR * 4

def chunk_file(src_path, label, speaker_id, source_name):
    rows = []
    audio = load_mono_16k(src_path)
    n_chunks = len(audio) // CHUNK_SAMPLES
    stem = Path(src_path).stem
    for i in range(n_chunks):
        chunk = audio[i * CHUNK_SAMPLES:(i + 1) * CHUNK_SAMPLES]
        out_path = f'{DEEPVOICE_CHUNK_DIR}/{stem}_{i}.wav'
        if not os.path.exists(out_path):
            sf.write(out_path, chunk, TARGET_SR)
        rows.append({'path': out_path, 'label': label, 'source': source_name,
                     'speaker_id': speaker_id, 'split_hint': None})
    return rows

deepvoice_rows = []
real_files = glob.glob(f'{deepvoice_dir}/**/AUDIO/REAL/*.wav', recursive=True)
fake_files = glob.glob(f'{deepvoice_dir}/**/AUDIO/FAKE/*.wav', recursive=True)
print(f'DEEP-VOICE: {len(real_files)} real source files, {len(fake_files)} fake source files')
assert real_files and fake_files, 'Expected files under AUDIO/REAL and AUDIO/FAKE.'

for f in tqdm(real_files, desc='chunking real'):
    speaker = Path(f).stem.replace('-original', '')
    deepvoice_rows += chunk_file(f, 'genuine', speaker, 'deepvoice_real')

for f in tqdm(fake_files, desc='chunking fake'):
    pair = Path(f).stem
    source_speaker = pair.split('-to-')[0].lower()
    deepvoice_rows += chunk_file(f, 'deepfake', source_speaker, 'deepvoice_fake')

dv_df_check = pd.DataFrame(deepvoice_rows)
print(f'\nDEEP-VOICE chunked rows: {len(deepvoice_rows)}')
print(dv_df_check.groupby('label').size())


## Upload realistic eval dataset

In [ ]:
from google.colab import files
import zipfile

print('Upload the zipped "audio" folder (voice_changing dataset).')
uploaded = files.upload()
zip_name = next(iter(uploaded))

eval_dir = '/content/real_eval_audio'
os.makedirs(eval_dir, exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(eval_dir)
print('Extracted to', eval_dir)


In [ ]:
import glob

eval_files = (
    glob.glob(f'{eval_dir}/**/*.mp3', recursive=True) +
    glob.glob(f'{eval_dir}/**/*.m4a', recursive=True) +
    glob.glob(f'{eval_dir}/**/*.wav', recursive=True)
)
print(f'Found {len(eval_files)} real-world eval files.')
assert eval_files, 'No audio files found in the extracted zip -- check its folder structure.'


In [ ]:
from pathlib import Path

HELD_OUT_SPEAKER = ('USA', 'male')  # never used in training -- pure held-out eval target

realistic_finetune_rows = []
realistic_heldout_rows = []
for path in eval_files:
    parts = Path(path).parts
    country, gender = parts[-4], parts[-3]
    stem = Path(path).stem
    label = 'genuine' if stem == 'original' else 'deepfake'
    row = {'path': path, 'label': label, 'source': 'realistic_voice_changing',
           'speaker_id': f'{country}_{gender}', 'split_hint': None}
    if (country, gender) == HELD_OUT_SPEAKER:
        realistic_heldout_rows.append(row)
    else:
        realistic_finetune_rows.append(row)

print(f'Fine-tune rows (3 speakers): {len(realistic_finetune_rows)}')
print(f'Held-out eval rows (speaker {HELD_OUT_SPEAKER}): {len(realistic_heldout_rows)}')
assert realistic_finetune_rows and realistic_heldout_rows, 'Expected data on both sides of the speaker split.'


## Combine into the v6 manifest -- genuine realistic rows oversampled separately

The actual fix: split the realistic fine-tune rows by label BEFORE oversampling, so genuine
gets enough extra weight to land at roughly the same absolute count as deepfake, instead of
both getting the same multiplier applied to very different starting counts.

In [ ]:
import numpy as np

df_v2_base = pd.read_csv(f'{MANIFEST_DIR}/phase1_manifest.csv')
print('v2 base:')
print(df_v2_base.groupby(['split', 'label']).size())

MAX_PER_CLASS_V6 = 10000

realistic_genuine_rows = [r for r in realistic_finetune_rows if r['label'] == 'genuine']
realistic_deepfake_rows = [r for r in realistic_finetune_rows if r['label'] == 'deepfake']

DEEPFAKE_OVERSAMPLE_V6 = 20   # unchanged from v4/v5
GENUINE_OVERSAMPLE_V6 = 60    # 3x higher -- genuine started with ~1/3 as many raw rows
                               # (1 'original' vs 3 'synthetic_*' per script)

genuine_oversampled_v6 = pd.DataFrame(realistic_genuine_rows)
genuine_oversampled_v6 = pd.concat([genuine_oversampled_v6] * GENUINE_OVERSAMPLE_V6, ignore_index=True)

deepfake_oversampled_v6 = pd.DataFrame(realistic_deepfake_rows)
deepfake_oversampled_v6 = pd.concat([deepfake_oversampled_v6] * DEEPFAKE_OVERSAMPLE_V6, ignore_index=True)

print(f'Realistic genuine: {len(realistic_genuine_rows)} raw -> {len(genuine_oversampled_v6)} after {GENUINE_OVERSAMPLE_V6}x')
print(f'Realistic deepfake: {len(realistic_deepfake_rows)} raw -> {len(deepfake_oversampled_v6)} after {DEEPFAKE_OVERSAMPLE_V6}x')

realistic_oversampled_v6 = pd.concat([genuine_oversampled_v6, deepfake_oversampled_v6], ignore_index=True)

new_source_rows = adf_rows + deepvoice_rows + realistic_oversampled_v6.to_dict('records')
new_df = pd.DataFrame(new_source_rows)
new_df['split'] = 'train'  # all new data goes to train; v2's original val/test stay untouched

all_v6 = pd.concat([df_v2_base, new_df], ignore_index=True)
print('\nCombined, before re-capping:')
print(all_v6.groupby(['label']).size())

def cap_group_v6(g):
    if len(g) <= MAX_PER_CLASS_V6:
        return g
    parts = []
    sources = g['source'].unique()
    budget_per_source = MAX_PER_CLASS_V6 // len(sources)
    for s in sources:
        sub = g[g['source'] == s]
        parts.append(sub.sample(min(len(sub), budget_per_source), random_state=42))
    capped = pd.concat(parts)
    if len(capped) < MAX_PER_CLASS_V6:
        remaining = g.drop(capped.index)
        top_up = remaining.sample(min(len(remaining), MAX_PER_CLASS_V6 - len(capped)), random_state=42)
        capped = pd.concat([capped, top_up])
    return capped

df_v6 = all_v6.groupby('label', group_keys=False).apply(cap_group_v6).reset_index(drop=True)
print('\nAfter capping to', MAX_PER_CLASS_V6, 'per class (proportional across sources):')
print(df_v6.groupby(['label']).size())
print('\nSource breakdown per class:')
print(df_v6.groupby(['label', 'source']).size())
print('\nFinal split sizes:')
print(df_v6.groupby(['split', 'label']).size())

manifest_path_v6 = f'{MANIFEST_DIR_V6}/v6_manifest.csv'
df_v6.to_csv(manifest_path_v6, index=False)
print('\nSaved to', manifest_path_v6)


## Train v6 -- same gentle fine-tune-from-v2 recipe as v5, rebalanced data

In [ ]:
train_ds_v6 = AudioDataset(df_v6, 'train', train=True)
val_ds_v6 = AudioDataset(df_v6, 'val', train=False)
test_ds_v6 = AudioDataset(df_v6, 'test', train=False)
print(f'train={len(train_ds_v6)}  val={len(val_ds_v6)}  test={len(test_ds_v6)}')

v6_model = AudioClassifier(num_classes=len(LABELS)).to(device)
v6_model.load_state_dict(torch.load(f'{export_dir}/wav2vec2_deepfake_classifier.pt', map_location=device))
print('Loaded v2 weights as the v6 fine-tuning starting point.')


In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from tqdm import tqdm
import csv, os, shutil

V6_BATCH_SIZE = 8
V6_GRAD_ACCUM = 4
V6_EPOCHS = 5
V6_LR = 1e-5        # same gentle rate as v5 -- only the data changed, not the recipe

def collate(batch):
    waveforms = torch.stack([b[0] for b in batch])
    labels = torch.tensor([b[1] for b in batch])
    return waveforms, labels

v6_train_loader = DataLoader(train_ds_v6, batch_size=V6_BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=2)
v6_val_loader = DataLoader(val_ds_v6, batch_size=V6_BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=2)

train_counts = df_v6[df_v6['split'] == 'train']['label'].value_counts()
weights = torch.tensor([1.0 / train_counts.get(l, 1) for l in LABELS], dtype=torch.float32)
weights = weights / weights.sum() * len(LABELS)
weights = weights.to(device)
print('Class weights:', dict(zip(LABELS, weights.tolist())))

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = AdamW(v6_model.parameters(), lr=V6_LR)
total_steps = (len(v6_train_loader) // V6_GRAD_ACCUM) * V6_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * total_steps), total_steps)

def evaluate_v6(loader):
    v6_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for waveforms, labels in loader:
            waveforms = waveforms.to(device)
            logits = v6_model(waveforms)
            all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
            all_labels.extend(labels.numpy())
    return f1_score(all_labels, all_preds, average='macro')

LOCAL_CKPT_DIR_V6 = '/content/checkpoints_v6_local'
os.makedirs(LOCAL_CKPT_DIR_V6, exist_ok=True)

def save_ckpt_verified_v6(obj, filename):
    local_path = f'{LOCAL_CKPT_DIR_V6}/{filename}'
    drive_path = f'{CKPT_DIR_V6}/{filename}'
    torch.save(obj, local_path)
    local_size = os.path.getsize(local_path)
    shutil.copy2(local_path, drive_path)
    drive_ok = os.path.exists(drive_path) and os.path.getsize(drive_path) == local_size
    print(f'  [ckpt] {filename}: local={local_size/1e6:.1f}MB  drive_saved={drive_ok}')
    if not drive_ok:
        print(f'  [ckpt] WARNING: Drive copy failed for {filename} -- local fallback at {local_path}')
    return drive_ok

ckpt_path_v6 = f'{CKPT_DIR_V6}/latest.pt'
log_path_v6 = f'{LOG_DIR_V6}/training_log.csv'
start_epoch = 0
best_val_f1 = -1
epochs_no_improve = 0
PATIENCE = 2

pre_ft_val_f1 = evaluate_v6(v6_val_loader)
best_val_f1 = pre_ft_val_f1
print(f'val macro-F1 BEFORE v6 fine-tuning: {pre_ft_val_f1:.4f} (should roughly match v2\'s own val F1)')

if os.path.exists(ckpt_path_v6):
    state = torch.load(ckpt_path_v6, map_location=device)
    v6_model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    start_epoch = state['epoch'] + 1
    best_val_f1 = state.get('best_val_f1', pre_ft_val_f1)
    epochs_no_improve = state.get('epochs_no_improve', 0)
    print(f'Resuming v6 training from epoch {start_epoch} (best so far: {best_val_f1:.4f})')
else:
    print('No checkpoint found -- starting v6 fresh from epoch 0.')

if not os.path.exists(log_path_v6):
    with open(log_path_v6, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch', 'train_loss', 'val_macro_f1'])

for epoch in range(start_epoch, V6_EPOCHS):
    v6_model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    for step, (waveforms, labels) in enumerate(tqdm(v6_train_loader, desc=f'v6 epoch {epoch}')):
        waveforms, labels = waveforms.to(device), labels.to(device)
        logits = v6_model(waveforms)
        loss = criterion(logits, labels) / V6_GRAD_ACCUM
        loss.backward()
        running_loss += loss.item() * V6_GRAD_ACCUM
        if (step + 1) % V6_GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(v6_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    train_loss = running_loss / len(v6_train_loader)
    val_f1 = evaluate_v6(v6_val_loader)
    print(f'epoch {epoch}: train_loss={train_loss:.4f}  val_macro_f1={val_f1:.4f}')
    with open(log_path_v6, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, train_loss, val_f1])

    improved = val_f1 > best_val_f1
    if improved:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        save_ckpt_verified_v6(v6_model.state_dict(), 'best.pt')
    else:
        epochs_no_improve += 1

    save_ckpt_verified_v6({
        'model': v6_model.state_dict(), 'optimizer': optimizer.state_dict(),
        'epoch': epoch, 'best_val_f1': best_val_f1, 'epochs_no_improve': epochs_no_improve,
    }, 'latest.pt')

    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

print(f'\nv6 training done. val macro-F1: before={pre_ft_val_f1:.4f}, best after={best_val_f1:.4f}')
print(f'Local fallback dir (this session only): {LOCAL_CKPT_DIR_V6}')


## Verify the v6 checkpoint landed on Drive before evaluating

In [ ]:
print(sorted(os.listdir(CKPT_DIR_V6)))


## Evaluate v6 against the held-out speaker (v2 vs v3 vs v4 vs v5 vs v6)

If `best.pt` doesn't exist (same situation as v5, if no epoch beats the pre-training
baseline), load `checkpoints_v6/latest.pt` instead and pull out its `['model']` key, same
workaround as before.

In [ ]:
best_state_v6 = torch.load(f'{CKPT_DIR_V6}/best.pt', map_location=device)
v6_model.load_state_dict(best_state_v6)
v6_model.eval()

rows = []
for r in realistic_heldout_rows:
    true_label = r['label']
    v2_probs = infer_with(eval_model, r['path'])
    v3_probs = infer_with(ft_model, r['path'])
    v4_probs = infer_with(v4_model, r['path'])
    v5_probs = infer_with(v5_model, r['path'])
    v6_probs = infer_with(v6_model, r['path'])
    rows.append({
        'file': Path(r['path']).parent.name + '/' + Path(r['path']).stem,
        'true_label': true_label,
        'v2_pred': EVAL_LABELS[v2_probs.argmax()],
        'v3_pred': EVAL_LABELS[v3_probs.argmax()],
        'v4_pred': EVAL_LABELS[v4_probs.argmax()],
        'v5_pred': EVAL_LABELS[v5_probs.argmax()],
        'v6_pred': EVAL_LABELS[v6_probs.argmax()],
        'v6_genuine_p': v6_probs[0], 'v6_ai_generated_p': v6_probs[1], 'v6_deepfake_p': v6_probs[2],
    })

comparison_v6_df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', None)
print(comparison_v6_df.to_string(index=False))

for col in ['v2_pred', 'v3_pred', 'v4_pred', 'v5_pred', 'v6_pred']:
    acc = (comparison_v6_df['true_label'] == comparison_v6_df[col]).mean()
    print(f'{col} accuracy on held-out speaker: {acc:.1%}')

comparison_v6_df.to_csv(f'{PROJECT_DIR}/v2_v3_v4_v5_v6_heldout_comparison.csv', index=False)


## Export v6

In [ ]:
export_dir_v6 = f'{PROJECT_DIR}/export_{RUN_NAME_V6}'
os.makedirs(export_dir_v6, exist_ok=True)
torch.save(v6_model.state_dict(), f'{export_dir_v6}/wav2vec2_deepfake_classifier.pt')
import json as json_lib
with open(f'{export_dir_v6}/label_map.json', 'w') as f:
    json_lib.dump({'labels': LABELS, 'model_name': 'facebook/wav2vec2-base',
                   'note': 'v6 = v2 fine-tuned (lr=1e-5) on v5\'s data with genuine realistic '
                           'rows oversampled separately (60x vs deepfake\'s 20x) to fix the '
                           '1-original-vs-3-synthetic folder imbalance'},
                  f, indent=2)
print('Exported v6 to', export_dir_v6)
